# 06 — Predictive model: exploration

Every model, before choosing one (notebook 07 has the final figures).

| plot | was | what |
|---|---|---|
| 6.0 | 10.2 | the four rankings, Cam-CAN (bstm / fcm, rest / movie) |
| 6.1 | 10.3 | which modelling choices matter, Cam-CAN |
| 6.2 | 10.5 | what the best models lean on (SHAP shares), Cam-CAN |
| 6.3 | 13.2 | the rankings, Cam-CAN → HCP |
| 6.4 | 13.3 | which modelling choices matter, Cam-CAN → HCP |
| 6.5 | new | what the best transferred models lean on (SHAP on the HCP subjects) |

The SHAP shares (6.2, 6.5) are read from cache: `sbatch slurm/predictive_cache.sbatch`.


## Part 1 — Cam-CAN, cross-validated

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from fmri_decomposition.supervised_model import hads as ht
from fmri_decomposition.supervised_model import report as rp
from fmri_decomposition.supervised_model import transfer_tables as tt
from fmri_decomposition.figures import hads as hp

# Cam-CAN, notebook 10's settings (report.CAM); override any of them here, e.g. save_dir="figures/07"
cfg = rp.cam_settings()
st = ht.Study(cfg)
T0, T1 = cfg.tasks
print(f"phenotype: {len(st.pheno):,} subjects;",
      ", ".join(f"{lab} {st.pheno[lab].value_counts().sort_index().astype(int).to_dict()}" for lab in cfg.targets))


### Plot 6.0 — the four rankings (was 10.2)

In [ ]:
base = st.baselines()
rows = st.performance_rows(base)
hp.fig_rankings(rows, cfg, st.above_both)
st.above_table(rows)


### Plot 6.1 — which modelling choices matter (was 10.3)

In [ ]:
hp.fig_factors(rows, cfg)


### Plot 6.2 — what the best models lean on (was 10.5)

In [ ]:
contrib = st.shares()          # cached
hp.fig_shares(contrib, cfg, f"the all arms: which variable type the model leans on (top {cfg.share_top} per model and arm)")


## Part 2 — trained on all of Cam-CAN, tested on HCP

In [ ]:
# Cam-CAN -> HCP, notebook 13's setup (report.transfer_setup)
TARGET = "category"      # "raw" | "category" (the manuscript)
train, test, train_cfg, test_cfg, ROOT = rp.transfer_setup(TARGET)
raw_rows, rows_t = rp.transferred_rows(ROOT, TARGET)
print(f"{len(rows_t)} transferred configuration(s) over {rows_t['states'].nunique()} state set(s); rows from {ROOT}")


### Plot 6.3 — the rankings, transferred (was 13.2)

In [ ]:
hp.fig_rankings(rows_t, train_cfg, train.above_both, fallback=train.above_zero)


### Plot 6.4 — which modelling choices matter, transferred (was 13.3)

In [ ]:
hp.fig_factors(rows_t, train_cfg)


### Plot 6.5 — what the best transferred models lean on (new, 6.2's analogue)

The best transferred configurations per (task, phenotype, model, arm), each fitted once on all of Cam-CAN; the share of its mean |SHAP| **on the HCP subjects** that each variable type takes. The score beside each row is the transferred Spearman ρ on HCP.

In [ ]:
contrib_t = rp.transfer_contrib(train, test, rows_t, arms=rp.transfer_share_arms(train_cfg, rows_t))   # cached
hp.fig_shares(contrib_t, train_cfg, f"transferred: which variable type the model leans on (top {train_cfg.share_top} per model and arm)",
              xlabel="share of the model's mean |SHAP| on the HCP subjects (one fit on all of Cam-CAN)",
              name="C_shares_transfer", score="ρ HCP")
